<a href="https://colab.research.google.com/github/TheGit-Father/cuda-matmul/blob/main/vector_add.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi

Sat Jul 25 03:36:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install numba

In [3]:
import numpy as np
from numba import cuda

#CUDA Kernel
@cuda.jit
def vector_add(a,b,c):
  idx=cuda.grid(1) #global thread index

  if idx<c.size:
    c[idx]=a[idx]+b[idx]

#input arrays
n=1000000

a=np.random.rand(n).astype(np.float32)
b=np.random.rand(n).astype(np.float32)

#output array
c=np.zeros(n, dtype=np.float32)

#Copy data to GPU
d_a=cuda.to_device(a)
d_b=cuda.to_device(b)
d_c=cuda.device_array_like(c)

#Configure threads
threads_per_block=256
blocks_per_grid=(n+threads_per_block-1)//threads_per_block

#Launch Kernel
vector_add[blocks_per_grid, threads_per_block](d_a,d_b,d_c)

#Copy result back
c=d_c.copy_to_host()

#Verify
print("Correct", np.allclose(c,a+b))
print("First 5 results")
print(c[:5])



Correct True
First 5 results
[0.81028056 1.0609798  0.33029172 0.59424263 1.2197313 ]
